# Invoice Region Detection and Business Parameter Extraction Using CNN, SSD, IoU, OCR, and Streamlit

**Member:** Rolando
**Role:** Data Ingestion, Dataset Management, Data Preparation Lead

**Objective:** Download/load the invoice dataset(s), validate and clean the images, split into train/val/test, apply preprocessing, and produce the manifest + quality report that every other notebook depends on.

**Inputs expected:**
- Kaggle datasets (see `../../dataset_sources.md`), downloaded to `data/raw/`

**Outputs generated:**
- `data/processed/invoice_manifest.csv`
- `outputs/reports/data_quality_report.md`
- `outputs/figures/sample_invoice_grid.png`
- `outputs/figures/preprocessing_examples.png`

> Run this notebook top-to-bottom in Google Colab, or locally with the repo's virtualenv.
> Paths are resolved via `src/config.py` (pathlib-based) — never hardcode absolute local paths.


In [ ]:
# --- Google Colab setup cell ---
# If running in Colab: clone the repo (or mount Drive if you cloned there already) and
# install dependencies. Safe to skip locally if the repo is already on disk with deps installed.

import sys, os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/<your-org>/invoice-image-processing.git"  # TODO: set this
    REPO_DIR = "/content/invoice-image-processing"

    if not os.path.exists(REPO_DIR):
        os.system(f"git clone {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)
    os.system("pip install -q -r requirements.txt")

    # Kaggle API credentials (upload kaggle.json when prompted) -- see dataset_sources.md
    from google.colab import files
    if not os.path.exists("/root/.kaggle/kaggle.json"):
        print("Upload your kaggle.json (Kaggle -> Account -> Create New API Token):")
        uploaded = files.upload()
        os.makedirs("/root/.kaggle", exist_ok=True)
        for fname in uploaded:
            os.replace(fname, "/root/.kaggle/kaggle.json")
        os.chmod("/root/.kaggle/kaggle.json", 0o600)

    print("Colab environment ready. Working directory:", os.getcwd())
else:
    print("Not running in Colab -- assuming local repo checkout with requirements installed.")


In [ ]:
# --- Dataset path setup cell ---
# All paths go through src.config.PATHS (pathlib-based, no hardcoded absolute paths).

import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "requirements.txt").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import PATHS, load_label_schema, load_required_fields

print("Repo root:", PATHS.repo_root)
print("Raw data dir:", PATHS.raw_dir)
print("Outputs dir:", PATHS.outputs_dir)

# If raw data isn't present yet, download it (see dataset_sources.md for kaggle.json setup):
#   python scripts/download_datasets.py --dataset all


In [ ]:
# --- Imports cell ---
import json

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from src.image_preprocessing import preprocess_pipeline, to_grayscale, resize_image, denoise_image, threshold_image, deskew_image
from src.visualization import draw_boxes, show_image_grid
from src.annotation_utils import load_annotations, boxes_for_image
from src.iou import compute_iou, precision_recall_iou, evaluate_predictions_df


## 1. Download / locate raw invoice images

In [ ]:
# If data/raw/invoices is empty, download it first:
#   python scripts/download_datasets.py --dataset invoices
from src.data_loader import list_raw_images

image_paths = list_raw_images("invoices")
print(f"Found {len(image_paths)} candidate invoice images.")


## 2. Validate folder paths & read image files

In [ ]:
# TODO: confirm data/raw/invoices exists and is non-empty (fail loudly if not,
# with a message pointing at scripts/download_datasets.py).
assert len(image_paths) > 0, "No images found -- run scripts/download_datasets.py --dataset invoices"


## 3. Build the invoice manifest (document_id, path, dims, file type, corrupt flag)

In [ ]:
records = []
for path in tqdm(image_paths):
    try:
        img = cv2.imread(str(path))
        is_corrupt = img is None
        h, w = (img.shape[0], img.shape[1]) if img is not None else (None, None)
    except Exception:
        is_corrupt, h, w = True, None, None

    records.append({
        "document_id": path.stem,
        "image_path": str(path.relative_to(PATHS.repo_root)),
        "width": w,
        "height": h,
        "file_type": path.suffix.lower(),
        "is_corrupt": is_corrupt,
        "split": None,  # filled in below
    })

manifest = pd.DataFrame(records)
print(manifest["is_corrupt"].value_counts())
manifest.head()


## 4. Flag/remove corrupt images, then split into train/val/test (e.g. 70/15/15)

In [ ]:
clean = manifest[~manifest["is_corrupt"]].reset_index(drop=True)

import numpy as np
rng = np.random.default_rng(42)
splits = rng.choice(["train", "val", "test"], size=len(clean), p=[0.7, 0.15, 0.15])
clean["split"] = splits

manifest = pd.concat([clean, manifest[manifest["is_corrupt"]]], ignore_index=True)
manifest["split"] = manifest["split"].fillna("excluded")
manifest["split"].value_counts()


## 5. Preprocessing (grayscale, resize, denoise, threshold, deskew) on a working subset

In [ ]:
SAMPLE_SIZE = 20  # keep small for a fast demo; raise if compute allows

sample_ids = clean.sample(min(SAMPLE_SIZE, len(clean)), random_state=42)["document_id"].tolist()
before_after = []

for doc_id in sample_ids:
    row = manifest.loc[manifest["document_id"] == doc_id].iloc[0]
    img = cv2.imread(str(PATHS.repo_root / row["image_path"]))
    processed = preprocess_pipeline(img, do_grayscale=True, do_denoise=True, do_deskew=True, do_threshold=False)
    before_after.append((img, processed))

    out_path = PATHS.processed_dir / f"{doc_id}.png"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(out_path), processed)

print(f"Preprocessed and saved {len(before_after)} images to {PATHS.processed_dir}")


## 6. Save sample visualizations

In [ ]:
from src.visualization import show_image_grid

PATHS.figures_dir.mkdir(parents=True, exist_ok=True)

sample_imgs = [cv2.imread(str(PATHS.repo_root / manifest.loc[manifest['document_id'] == d, 'image_path'].iloc[0])) for d in sample_ids[:12]]
show_image_grid(sample_imgs, titles=sample_ids[:12], cols=4, save_path=PATHS.figures_dir / "sample_invoice_grid.png")

pre_post_imgs, pre_post_titles = [], []
for img, processed in before_after[:6]:
    pre_post_imgs += [img, processed]
    pre_post_titles += ["original", "preprocessed"]
show_image_grid(pre_post_imgs, titles=pre_post_titles, cols=4, save_path=PATHS.figures_dir / "preprocessing_examples.png")


## 7. Data quality report

In [ ]:
report_lines = [
    "# Data Quality Report",
    "",
    f"- Total images found: {len(manifest)}",
    f"- Corrupt/unreadable images: {int(manifest['is_corrupt'].sum())}",
    f"- Clean images: {len(clean)}",
    f"- Split sizes: {clean['split'].value_counts().to_dict()}",
    f"- Width range: {clean['width'].min()} - {clean['width'].max()}",
    f"- Height range: {clean['height'].min()} - {clean['height'].max()}",
    f"- File types: {clean['file_type'].value_counts().to_dict()}",
]
report_text = "\n".join(report_lines)
print(report_text)


In [ ]:
# --- Final export cell ---
# Save every output required by model_interface_contract.md

PATHS.processed_dir.mkdir(parents=True, exist_ok=True)
manifest.to_csv(PATHS.processed_dir / "invoice_manifest.csv", index=False)

PATHS.reports_dir.mkdir(parents=True, exist_ok=True)
(PATHS.reports_dir / "data_quality_report.md").write_text(report_text, encoding="utf-8")

# Mirror into this member's own outputs/ folder too
member_out = PATHS.member_outputs_dir("rolando_data_ingestion")
member_out.mkdir(parents=True, exist_ok=True)
manifest.to_csv(member_out / "invoice_manifest.csv", index=False)
(member_out / "data_quality_report.md").write_text(report_text, encoding="utf-8")

print('Export complete.')
